# Ejercicio Guiado: Pipeline de Clasificacion con Prefect + MLflow

## Objetivo

En este ejercicio vas a construir, **paso a paso**, un pipeline de Machine Learning completo que:

1. **Carga** un dataset de clasificacion (Iris)
2. **Preprocesa** las features (escalado)
3. **Entrena** un modelo (RandomForest)
4. **Registra** todo en MLflow (parametros, metricas, modelo)
5. **Orquesta** todo con Prefect (tasks y flows)

---

### Analogia: Linea de produccion

Piensa en una **fabrica de galletas**:

| Fabrica de galletas | Nuestro pipeline |
|---|---|
| Recibir ingredientes | Cargar datos |
| Mezclar y preparar | Preprocesar features |
| Hornear | Entrenar modelo |
| Control de calidad | Evaluar metricas |
| Etiquetar y registrar | Registrar en MLflow |
| **Gerente de planta** | **Prefect (orquestador)** |

El **gerente de planta (Prefect)** se asegura de que cada paso se ejecute en orden, registra que paso va, y avisa si algo falla.

El **control de calidad (MLflow)** registra las recetas usadas (parametros), los resultados (metricas) y guarda el producto (modelo).

---

### Que dataset vamos a usar?

El **dataset Iris** es el "Hola Mundo" del Machine Learning:
- 150 flores medidas
- 4 caracteristicas: largo/ancho de sepalo y petalo
- 3 clases: Setosa, Versicolor, Virginica
- Viene incluido en scikit-learn (no hay que descargar nada)

---

### Diferencias con el pipeline de NYC Taxi que vimos en clase

| Pipeline NYC Taxi (clase) | Este ejercicio |
|---|---|
| Regresion (predecir duracion) | **Clasificacion** (predecir especie) |
| Datos de internet (parquet) | Datos incluidos en sklearn |
| XGBoost + Optuna | **RandomForest** (sin optimizacion) |
| DictVectorizer | **StandardScaler** |
| RMSE | **Accuracy** |
| Muchos archivos y logica | **Menos archivos, mas simple** |

La **estructura modular es la misma** — eso es lo importante.

---

## Paso 0: Prerrequisitos

Asegurate de tener instaladas estas librerias. Si no las tienes, ejecuta la celda de abajo:

In [ ]:
# Descomentar y ejecutar si necesitas instalar
# !pip install prefect mlflow scikit-learn

---

## Paso 1: Crear la estructura del proyecto

Todo buen proyecto de ML tiene una **estructura organizada**. Vamos a crear esta:

```
ejercicio/
├── ejercicio_pipeline_clasificacion.ipynb   <-- Este notebook (ya lo tienes)
├── pipeline.py          <-- Flow principal (Prefect)
└── src/                 <-- Modulos del pipeline
    ├── __init__.py      <-- Hace que src/ sea un paquete Python
    ├── config.py        <-- Constantes y configuracion
    ├── data.py          <-- Carga y division de datos
    ├── features.py      <-- Preprocesamiento
    └── models.py        <-- Entrenamiento y registro en MLflow
```

### Por que esta estructura?

- **`config.py`**: Un solo lugar para cambiar parametros (no hardcodear numeros por todo el codigo)
- **`data.py`**: Separar la carga de datos del entrenamiento
- **`features.py`**: El preprocesamiento es un paso independiente
- **`models.py`**: El entrenamiento tiene su propia logica
- **`pipeline.py`**: Solo conecta los pasos — no tiene logica de negocio

Ejecuta la siguiente celda para crear la carpeta `src/`:

In [ ]:
import os
os.makedirs("src", exist_ok=True)
print("Carpeta src/ creada!")

---

## Paso 2: Crear el archivo de configuracion (`src/config.py`)

Este archivo centraliza **todas las constantes** del proyecto. Si quieres cambiar el numero de arboles del modelo o el tamanio del test set, solo modificas este archivo.

### Que contiene:
- **MLFLOW_EXPERIMENT_NAME**: Nombre del experimento en MLflow (como una carpeta donde se guardan los runs)
- **MLFLOW_TRACKING_URI**: Donde MLflow guarda los datos (usamos SQLite, un archivo local)
- **TEST_SIZE**: Proporcion de datos para test (0.2 = 20%)
- **RANDOM_STATE**: Semilla para reproducibilidad
- **N_ESTIMATORS**: Numero de arboles en el RandomForest
- **MAX_DEPTH**: Profundidad maxima de cada arbol

Ejecuta la celda para crear el archivo:

In [ ]:
%%writefile src/config.py
"""
Configuracion del pipeline de clasificacion.
Aqui centralizamos todas las constantes y parametros.
"""

# --- MLflow ---
MLFLOW_EXPERIMENT_NAME = "iris-clasificacion-prefect"
MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

# --- Datos ---
TEST_SIZE = 0.2
RANDOM_STATE = 42

# --- Modelo (RandomForest) ---
N_ESTIMATORS = 100
MAX_DEPTH = 5

Verifica que se creo correctamente:

In [ ]:
# Verificacion rapida
with open("src/config.py") as f:
    print(f.read())

---

## Paso 3: Crear el modulo de datos (`src/data.py`)

Este modulo tiene **dos tareas de Prefect** (`@task`):

1. **`cargar_datos()`**: Carga el dataset Iris y lo convierte a DataFrame
2. **`dividir_datos()`**: Separa en train/test

### Conceptos clave:

- **`@task`**: Le dice a Prefect "esto es un paso del pipeline". Prefect lo monitorea, registra su estado, y puede reintentarlo si falla.
- **`get_run_logger()`**: Logger especial de Prefect que muestra mensajes en la UI de Prefect.
- **`load_iris()`**: Funcion de sklearn que trae el dataset Iris ya listo.
- **`train_test_split()`**: Divide los datos aleatoriamente en entrenamiento y prueba.

Ejecuta la celda:

In [ ]:
%%writefile src/data.py
"""
Carga y preparacion de datos.
Usamos el dataset Iris que viene incluido en scikit-learn.
"""

import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from prefect import task, get_run_logger

from .config import TEST_SIZE, RANDOM_STATE


@task(name="cargar_datos", description="Carga el dataset Iris desde sklearn")
def cargar_datos() -> pd.DataFrame:
    """Carga el dataset Iris y lo convierte a DataFrame."""
    logger = get_run_logger()

    iris = load_iris()
    df = pd.DataFrame(iris.data, columns=iris.feature_names)
    df["target"] = iris.target

    logger.info(f"Dataset cargado: {len(df)} registros, {df.shape[1] - 1} features")
    logger.info(f"Clases: {list(iris.target_names)}")
    return df


@task(name="dividir_datos", description="Divide en conjuntos de entrenamiento y prueba")
def dividir_datos(df: pd.DataFrame):
    """Separa features (X) del target (y) y divide en train/test."""
    logger = get_run_logger()

    X = df.drop("target", axis=1)
    y = df["target"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    logger.info(f"Train: {len(X_train)} registros | Test: {len(X_test)} registros")
    return X_train, X_test, y_train, y_test

### Nota sobre los imports relativos

Fijate que usamos `from .config import ...` (con un **punto** antes de `config`). Esto se llama **import relativo** y significa "importa desde el mismo paquete (`src/`)". Solo funciona dentro de paquetes Python (carpetas con `__init__.py`).

---

## Paso 4: Crear el modulo de features (`src/features.py`)

Aqui escalamos las features numericas usando **StandardScaler**.

### Por que escalar?

Imagina que mides personas:
- Altura: 150-200 cm
- Peso: 50-120 kg

Sin escalar, el modelo podria darle mas importancia a la altura solo porque tiene numeros mas grandes. El StandardScaler pone todo en la misma escala (media=0, desviacion=1).

### Regla de oro del escalado:
- **fit_transform** en train (aprende la escala Y transforma)
- **transform** en test (usa la misma escala, NO aprende de nuevo)

Si haces fit en test, estarias "haciendo trampa" (data leakage).

Ejecuta la celda:

In [ ]:
%%writefile src/features.py
"""
Preprocesamiento de features.
Escalamos las variables numericas para que tengan media 0 y desviacion estandar 1.
"""

from sklearn.preprocessing import StandardScaler
from prefect import task, get_run_logger


@task(name="escalar_features", description="Escala features con StandardScaler")
def escalar_features(X_train, X_test):
    """Aplica StandardScaler: fit en train, transform en ambos."""
    logger = get_run_logger()

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    logger.info(f"Features escaladas: {X_train_scaled.shape[1]} columnas")
    return X_train_scaled, X_test_scaled, scaler

---

## Paso 5: Crear el modulo de modelos (`src/models.py`)

Este es el modulo mas importante. Aqui:

1. **Entrenamos** un RandomForestClassifier
2. **Evaluamos** con accuracy
3. **Registramos todo en MLflow**:
   - `mlflow.log_param()` -> Guarda parametros (n_estimators, max_depth, etc.)
   - `mlflow.log_metric()` -> Guarda metricas (accuracy)
   - `mlflow.sklearn.log_model()` -> Guarda el modelo completo
   - `mlflow.log_artifact()` -> Guarda archivos adicionales (el scaler)

### Anatomia de un run de MLflow:

```
mlflow.start_run()          <-- Abre un "expediente"
    log_param(...)          <-- Anota la receta
    modelo.fit(...)         <-- Entrena
    log_metric(...)         <-- Anota el resultado
    log_model(...)          <-- Guarda el modelo
    log_artifact(...)       <-- Guarda archivos extra
                            <-- Se cierra automaticamente (with)
```

Ejecuta la celda:

In [ ]:
%%writefile src/models.py
"""
Entrenamiento del modelo y registro en MLflow.
"""

import pickle
from pathlib import Path

import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from prefect import task, get_run_logger

from .config import N_ESTIMATORS, MAX_DEPTH, RANDOM_STATE


@task(name="entrenar_modelo", description="Entrena RandomForest y registra en MLflow")
def entrenar_modelo(X_train, y_train, X_test, y_test, scaler):
    """Entrena un RandomForest, evalua y registra todo en MLflow."""
    logger = get_run_logger()

    with mlflow.start_run() as run:
        # 1. Registrar parametros
        mlflow.log_param("n_estimators", N_ESTIMATORS)
        mlflow.log_param("max_depth", MAX_DEPTH)
        mlflow.log_param("random_state", RANDOM_STATE)
        mlflow.log_param("modelo", "RandomForestClassifier")

        # 2. Entrenar modelo
        modelo = RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            max_depth=MAX_DEPTH,
            random_state=RANDOM_STATE,
        )
        modelo.fit(X_train, y_train)

        # 3. Evaluar
        y_pred = modelo.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        # 4. Registrar metricas
        mlflow.log_metric("accuracy", accuracy)

        # 5. Registrar modelo completo
        mlflow.sklearn.log_model(modelo, "modelo_random_forest")

        # 6. Guardar scaler como artefacto
        models_dir = Path("models")
        models_dir.mkdir(exist_ok=True)
        scaler_path = models_dir / "scaler.pkl"
        with open(scaler_path, "wb") as f:
            pickle.dump(scaler, f)
        mlflow.log_artifact(str(scaler_path))

        logger.info(f"Accuracy: {accuracy:.4f}")
        logger.info(f"MLflow Run ID: {run.info.run_id}")

        return run.info.run_id, accuracy

---

## Paso 6: Crear el archivo `__init__.py`

Este archivo le dice a Python que la carpeta `src/` es un **paquete** (un modulo importable). Sin este archivo, los imports `from src.data import ...` no funcionarian.

Puede estar vacio o tener un comentario simple:

In [ ]:
%%writefile src/__init__.py
# Modulos del pipeline de clasificacion

### Verificacion: revisa que todos los archivos estan creados

In [ ]:
import os

archivos_esperados = [
    "src/__init__.py",
    "src/config.py",
    "src/data.py",
    "src/features.py",
    "src/models.py",
]

print("Verificando archivos creados:")
print("=" * 40)
todos_ok = True
for archivo in archivos_esperados:
    existe = os.path.exists(archivo)
    estado = "OK" if existe else "FALTA"
    print(f"  {estado} -> {archivo}")
    if not existe:
        todos_ok = False

if todos_ok:
    print("\nTodos los modulos estan listos!")
else:
    print("\nAlgunos archivos faltan. Revisa los pasos anteriores.")

---

## Paso 7: Crear el pipeline principal (`pipeline.py`)

Este archivo es el **orquestador**. Usa el decorador `@flow` de Prefect para definir el flujo completo.

### Diferencia entre `@task` y `@flow`:

| | `@task` | `@flow` |
|---|---|---|
| **Que es** | Un paso individual | El proceso completo |
| **Analogia** | Un trabajador en la fabrica | El gerente de planta |
| **Contiene** | Logica especifica | Llamadas a tasks |
| **Ejemplo** | `cargar_datos()` | `clasificacion_flow()` |

El flow **no tiene logica de negocio**, solo conecta los tasks en el orden correcto.

Ejecuta la celda:

In [ ]:
%%writefile pipeline.py
"""
Pipeline principal de clasificacion Iris con Prefect + MLflow.
Orquesta todos los pasos: carga, division, features, entrenamiento.
"""

import mlflow
from prefect import flow, get_run_logger

from src.config import MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_URI
from src.data import cargar_datos, dividir_datos
from src.features import escalar_features
from src.models import entrenar_modelo


@flow(name="Iris Classification Pipeline", log_prints=True)
def clasificacion_flow():
    """Flow principal que conecta todos los pasos del pipeline."""
    logger = get_run_logger()

    # Configurar MLflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

    # Paso 1: Cargar datos
    df = cargar_datos()

    # Paso 2: Dividir en train/test
    X_train, X_test, y_train, y_test = dividir_datos(df)

    # Paso 3: Escalar features
    X_train_scaled, X_test_scaled, scaler = escalar_features(X_train, X_test)

    # Paso 4: Entrenar modelo y registrar en MLflow
    run_id, accuracy = entrenar_modelo(
        X_train_scaled, y_train, X_test_scaled, y_test, scaler
    )

    logger.info(f"Pipeline completado! Accuracy: {accuracy:.4f}")
    logger.info(f"MLflow Run ID: {run_id}")

    return run_id


if __name__ == "__main__":
    clasificacion_flow()

---

## Paso 8: Ejecutar el pipeline!

Ahora viene el momento de la verdad. Vamos a ejecutar el pipeline completo.

### Que va a pasar cuando ejecutes la siguiente celda:

1. Prefect inicia el flow `Iris Classification Pipeline`
2. Se ejecuta `cargar_datos` -> Carga las 150 flores del dataset Iris
3. Se ejecuta `dividir_datos` -> 120 para train, 30 para test
4. Se ejecuta `escalar_features` -> Normaliza las 4 columnas
5. Se ejecuta `entrenar_modelo` -> Entrena RandomForest, calcula accuracy, registra en MLflow
6. Prefect muestra el resumen del pipeline

Ejecuta la celda y observa los logs:

In [ ]:
# Importamos y ejecutamos el pipeline
from pipeline import clasificacion_flow

run_id = clasificacion_flow()

print(f"\nPipeline completado!")
print(f"MLflow Run ID: {run_id}")

### Que deberias ver en los logs:

```
INFO  - Created task run 'cargar_datos-...' for task 'cargar_datos'
INFO  - Dataset cargado: 150 registros, 4 features
INFO  - Clases: ['setosa', 'versicolor', 'virginica']
INFO  - Created task run 'dividir_datos-...' for task 'dividir_datos'
INFO  - Train: 120 registros | Test: 30 registros
INFO  - Created task run 'escalar_features-...' for task 'escalar_features'
INFO  - Features escaladas: 4 columnas
INFO  - Created task run 'entrenar_modelo-...' for task 'entrenar_modelo'
INFO  - Accuracy: 1.0000
INFO  - MLflow Run ID: abc123...
INFO  - Pipeline completado! Accuracy: 1.0000
```

El Iris es tan simple que probablemente obtengas **accuracy = 1.0** (100%). Eso es normal para este dataset.

---

## Paso 9: Ver los resultados en MLflow

MLflow guardo todo en una base de datos SQLite (`mlflow.db`). Para ver los resultados en una interfaz web, ejecuta este comando **en una terminal aparte**:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Luego abre en tu navegador: **http://localhost:5000**

### Que buscar en la UI de MLflow:

1. **Experimento**: `iris-clasificacion-prefect` (panel izquierdo)
2. **Run**: Haz clic en el run mas reciente
3. **Parameters**: Veras `n_estimators=100`, `max_depth=5`, etc.
4. **Metrics**: Veras `accuracy=1.0`
5. **Artifacts**: Veras el modelo guardado y el `scaler.pkl`

---

## Paso 10 (Opcional): Ejecutar desde la terminal

Tambien puedes ejecutar el pipeline directamente desde la terminal:

```bash
python pipeline.py
```

Esto funciona gracias al bloque `if __name__ == "__main__"` al final de `pipeline.py`.

---

## Paso 11 (Opcional): Experimentar con los parametros

Ahora que tienes el pipeline funcionando, prueba a cambiar los parametros en `src/config.py` y re-ejecutar.

**Ideas para experimentar:**

| Cambio | Que pasa |
|---|---|
| `N_ESTIMATORS = 10` | Menos arboles, mas rapido, quizas menos accuracy |
| `MAX_DEPTH = 2` | Arboles menos profundos, modelo mas simple |
| `TEST_SIZE = 0.5` | Mitad para test, menos datos de entrenamiento |

Cada vez que ejecutes, MLflow creara un **nuevo run** con los nuevos parametros. Asi puedes comparar resultados!

Para re-ejecutar despues de cambiar config, reinicia el kernel del notebook (para recargar los modulos) y ejecuta el Paso 8 de nuevo.

---

## Resumen

En este ejercicio construiste un pipeline de ML completo con:

- **Prefect** como orquestador (`@flow` y `@task`)
- **MLflow** como tracker de experimentos (`log_param`, `log_metric`, `log_model`)
- **Estructura modular** (config, data, features, models, pipeline)

### Diagrama del flujo:

```
clasificacion_flow (Prefect @flow)
    |
    |-- cargar_datos()        (@task) -> DataFrame con 150 flores
    |
    |-- dividir_datos()       (@task) -> X_train, X_test, y_train, y_test
    |
    |-- escalar_features()    (@task) -> X_train_scaled, X_test_scaled, scaler
    |
    |-- entrenar_modelo()     (@task) -> run_id, accuracy
    |       |
    |       |-- mlflow.log_param()     -> Guarda parametros
    |       |-- mlflow.log_metric()    -> Guarda accuracy
    |       |-- mlflow.log_model()     -> Guarda modelo
    |       |-- mlflow.log_artifact()  -> Guarda scaler
    |
    |-- return run_id
```

Este es exactamente el mismo patron que usamos en el pipeline de NYC Taxi, pero simplificado. Cuando entiendas este flujo, el pipeline complejo es solo "mas de lo mismo".